# Projekt: Kundenstimmung auf Twitter verstehen

## „Kundensupport auf Twitter“ untersuchen
- mehrstufigen KI-Workflow nutzen, um die Daten eingehend zu verstehen und zu analysieren:
- mit llm , um den Datensatz automatisch zu erkunden und zusammenzufassen.
- mit AutoViz fort , um die Daten visuell zu verstehen.
- mit Hugging-Face-Modell verwendet , um die Stimmung in den Tweets der Kunden zu analysieren.

## Datensatz: Kundensupport auf Twitter

### Dieser Datensatz bietet drei wesentliche Vorteile gegenüber anderen Konversationsdatensätzen:
- Fokussiert : Die Gespräche drehen sich um reale Probleme, die die Menschen gelöst haben möchten – verlorenes Gepäck, Abrechnungsprobleme, stornierte Flüge – wodurch die Daten einen klaren Zweck und eine klare Struktur erhalten.
- Natürlich : Die Sprache ist modern und wirkt authentisch, geschrieben von Menschen mit unterschiedlichem Hintergrund. Sie spiegelt wider, wie Kunden heute tatsächlich online kommunizieren.
- Kurz und bündig : Da Tweets kurz sind, wirken die Antworten authentischer und weniger einstudiert. Dies hilft Modellen, natürlicher zu lernen und unterstützt zudem eine effiziente Verarbeitung.

1. Technische Artefakte.
- Von LLM generierte Datensatzzusammenfassungen oder Explorationsergebnisse
- AutoViz-Visualisierungen, die wichtige Muster hervorheben
- Ergebnisse der Stimmungsanalyse, die mit einem Hugging-Face-Modell erzeugt wurden
2. Analytisches Denken
- Wie Zwillinge Ihr anfängliches Verständnis und Ihre analytische Ausrichtung geprägt haben
- Was AutoViz aufdeckte, war aus den Textzusammenfassungen allein nicht ersichtlich.
- Wie die Ergebnisse der Stimmungsanalyse frühere Erkenntnisse ergänzten oder in Frage stellten
3. Überlegungen zur KI-gestützten Analyse
- Wo KI-Tools die Exploration beschleunigten oder den manuellen Aufwand reduzierten
- Wo menschliche Interpretation noch unerlässlich war
- Stärken und Schwächen der Verwendung von LLMs für die Stimmungsanalyse

Schriftliche Erläuterungen sollten als Markdown-Zellen neben den Ausgaben eingefügt werden.
Ziel ist es, Einsicht, Urteilsvermögen und den effektiven Einsatz von Werkzeugen zu demonstrieren, nicht eine erschöpfende Analyse.

In [1]:
# INITIALISIERUNG & COMPATIBILITY PATCHES

# 1. SYSTEM & DATEI-MANAGEMENT (Packet-Priorität)
import spacy
import json
import importlib.util
import os
import yaml
import sys
import time
import subprocess
import threading
import logging
import warnings
import psutil
import gc
import requests
import multiprocessing
from pathlib import Path

# 2. DATENVERARBEITUNG & COMPATIBILITY (Scipy Patch)
import numpy as np
import pandas as pd

try:
    import scipy._lib.deprecation as sd
    if not hasattr(sd, '_sub_module_deprecation'):
        sd._sub_module_deprecation = lambda *args, **kwargs: None
except ImportError:
    pass

# 3. FORTSCHRITTSANZEIGE & VISUALISIERUNG
from tqdm.auto import tqdm
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython import get_ipython
    if get_ipython() is not None:
        get_ipython().run_line_magic('matplotlib', 'inline')
except:
    pass

# 4. AUTOMATISIERTE EDA & STATISTIK (ydata/sweetviz/autoviz)
try:
    from autoviz.AutoViz_Class import AutoViz_Class
    from ydata_profiling import ProfileReport
    import sweetviz as sv
except Exception as e:
    print(f"⚠️ EDA-Tools teilweise nicht geladen: {e}")

# 5. LLM-AGENTEN-STEUERUNG
from pandasai import SmartDataframe
from pandasai.llm import LLM
import re

# 6. LOGIK-ENGINE, QS-VALIDIERUNG & GRAPHEN (Der Richter-Stack)
from pydantic import BaseModel, Field, field_validator # Struktur-Validierung
import networkx as nx                                  # Graphen-Logik (Reihenfolge)
from typing import List, Optional, Dict, Any, Union    # Typsicherheit

# 7. AUTO-ML (H2O Integration - Enterprise Standard)
import h2o
from h2o.automl import H2OAutoML

# 8. NLP, ML & DEEP LEARNING (TFRecord & LSTM Focus)
import nltk
try:
    from nltk.corpus import stopwords
except:
    nltk.download('stopwords')
    from nltk.corpus import stopwords

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, BatchNormalization, Embedding, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)


In [2]:
# HIGH-END DATA LOADING & AUTOMATISCHE PAKET-SPLITTING LOGIK (99MB)
# 1. PFAD-ERMITTLUNG
current = Path.cwd()
while not (current / "data").exists():
    if current.parent == current:
        raise FileNotFoundError("Projekt-Root mit 'data' Ordner nicht gefunden!")
    current = current.parent

PROJECT_ROOT = current
DATA_DIR = PROJECT_ROOT / "data"

def _write_part_with_hard_limit(df, start, end, out_path, hard_limit_bytes):
    """
    Schreibt df[start:end] nach out_path.
    Verkleinert 'end' iterativ, bis das 99MB Limit exakt eingehalten wird.
    """
    end = max(start + 1, end)
    while True:
        df.iloc[start:end].to_parquet(out_path, index=False)
        size = out_path.stat().st_size
        if size <= hard_limit_bytes or (end - start) <= 1:
            return end, size
        shrink_ratio = (hard_limit_bytes / size) * 0.98
        end = start + max(1, int((end - start) * shrink_ratio))

def split_to_packages(df, target_dir):
    """Splittet große Dataframes in exakte <99MB Parquet-Parts."""
    target_dir.mkdir(parents=True, exist_ok=True)
    total_rows = len(df)
    limit_bytes = int(99.0 * 1024 * 1024)
    sample_size = min(10000, total_rows)
    tmp_path = target_dir / "_size_check.tmp"
    df.iloc[:sample_size].to_parquet(tmp_path)
    bytes_per_row = max(1.0, tmp_path.stat().st_size / sample_size)
    tmp_path.unlink()
    rows_est = int(limit_bytes / bytes_per_row)

    start, part = 0, 0
    while start < total_rows:
        out_path = target_dir / f"part_{part}.pak"
        end_guess = min(total_rows, start + rows_est)
        end_final, written_bytes = _write_part_with_hard_limit(df, start, end_guess, out_path, limit_bytes)
        rows_est = max(1, int((end_final - start) * (limit_bytes / written_bytes)))
        start, part = end_final, part + 1
    return part

class LivePacketLoader:
    """
    DER AUTOMATISCHE ÜBERSETZER: Verhält sich zu 100% wie ein DF.
    Leitet JEDEN Pandas-Befehl aktiv an die Disk weiter.
    Unterstützt das neue .pak Format (Einzeldatei & Split-Ordner).
    """
    def __init__(self, pkg_path, var_name):
        self.pkg_path = Path(pkg_path)
        self.var_name = var_name

        if self.pkg_path.is_dir():
            self.parts = sorted(self.pkg_path.glob("*.pak"))
        else:
            self.parts = [self.pkg_path] if self.pkg_path.exists() else []

        if self.parts:
            p0 = pd.read_parquet(self.parts[0])
            self.columns = p0.columns
            self.dtypes = p0.dtypes
            self._num_parts = len(self.parts)
            self._rows_p0 = len(p0)
            del p0
            gc.collect()
        else:
            print(f"⚠️ Verbindung fehlgeschlagen: Keine .pak Daten für {var_name} gefunden.")

    def _apply_smoothing(self, df):
        """
        INTERNE PROFI-GLÄTTUNG:
        Wird on-the-fly beim Laden jedes Pakets ausgeführt.
        Erstellt df_Cleaning, um den Originalzustand auf Disk zu schützen.
        """
        df_Cleaning = df.copy()
        for col in df_Cleaning.columns:
            if df_Cleaning[col].dtype == 'object':
                df_Cleaning[col] = df_Cleaning[col].fillna("").str.strip()
            else:
                df_Cleaning[col] = df_Cleaning[col].fillna(0)
        return df_Cleaning

    def run_automated_eda(self, sample_size=50000):
        """
        NEU: Die Analyse-Funktion direkt im Loader.
        Verhindert 1000-fache Code-Wiederholung & schont den RAM.
        """
        print(f"📊 Starte automatisierte Analyse für {self.var_name}...")
        df_sample = self.head(sample_size)

        try:
            from ydata_profiling import ProfileReport

            profile = ProfileReport(df_sample, title=f"EDA {self.var_name}", minimal=True)
            output_file = DATA_DIR / f"EDA_{self.var_name}.html"
            #profile.to_file(output_file)

            print(f"✅ Analyse fertig: {output_file.name}")
        except Exception as e:
            print(f"❌ EDA fehlgeschlagen: {e}")

        del df_sample
        gc.collect()

    def __getattr__(self, name):
        """
        Dynamischer Mapper für Pandas-Operationen.
        Unterscheidet zwischen 'callable' (Methoden) und Attributen (Properties).
        """
        if name.startswith('_'):
            raise AttributeError(f"'{type(self).__name__}' has no attribute '{name}'")

        # Der Wrapper wird nur für aufrufbare Methoden verwendet
        def wrapper(*args, **kwargs):
            results = []
            math_ops = ["mean", "sum", "std", "var", "median", "sem", "count", "min", "max"]

            for part in self.parts:
                raw_part = pd.read_parquet(part) # Nutzt Packet-Typen (Parquet)
                df_part = self._apply_smoothing(raw_part)

                if hasattr(df_part, name):
                    method = getattr(df_part, name)
                    # Core-Reserve Standard: Numerische Operationen erzwingen
                    if name in math_ops:
                        try:
                            res = method(*args, numeric_only=True, **kwargs)
                        except TypeError:
                            res = method(*args, **kwargs)
                    else:
                        res = method(*args, **kwargs)
                    results.append(res)

                del df_part, raw_part # RAM-Schutz

            gc.collect() # 1 Core frei Standard

            if not results: return None

            # Logik zur Zusammenführung (Aggregat-Bildung)
            if isinstance(results[0], (pd.Series, pd.DataFrame)):
                combined = pd.concat(results)
                if name in ["sum", "count"]: return combined.groupby(level=0).sum()
                if name == "max": return combined.groupby(level=0).max()
                if name == "min": return combined.groupby(level=0).min()
                return combined.groupby(level=0).mean(numeric_only=True)

            elif isinstance(results[0], (int, float, np.number)):
                if name == "max": return max(results)
                if name == "min": return min(results)
                if name in ["sum", "count"]: return sum(results)
                return sum(results) / len(results)

            return results[0]

        # LOGIK-CHECK: Ist das Attribut eine Methode oder eine Eigenschaft (shape, columns etc.)?
        t_raw = pd.read_parquet(self.parts[0])
        temp_check = self._apply_smoothing(t_raw)
        attr_in_pandas = getattr(temp_check, name)

        if callable(attr_in_pandas):
            res_attr = wrapper
        else:
            res_attr = attr_in_pandas

        del temp_check, t_raw
        gc.collect()
        return res_attr

    def __getitem__(self, key):
        """Erlaubt df['spalte'] oder df[['a', 'b']]."""
        raw = pd.read_parquet(self.parts[0])
        return self._apply_smoothing(raw)[key]

    def head(self, n=5):
        """Liefert echte Pandas-Tabelle für die Anzeige."""
        raw = pd.read_parquet(self.parts[0]).head(n)
        return self._apply_smoothing(raw)

    def _repr_html_(self):
        """Notebook-Tabellenansicht."""
        return pd.read_parquet(self.parts[0]).head(5)._repr_html_()

    def shape(self):
        """Gibt die Gesamtgröße zurück."""
        return (len(self), len(self.columns))

    def __len__(self):
        """Ermöglicht len(df) ohne Absturz."""
        return self._rows_p0 * self._num_parts

    def get_data(self, part=0):
        """Explizites Laden eines Pakets für ML-Zwecke."""
        if 0 <= part < len(self.parts): return pd.read_parquet(self.parts[part])
        return None

    def __repr__(self):
        return f"🔗 Live-Verbindung [{self.var_name}]: {self._num_parts} Pakete (RAM-Schutz aktiv)."

def initialize_data_infrastructure(live_mode=True):
    file_list = list(DATA_DIR.glob("*.csv"))
    limit_99mb = 99 * 1024 * 1024

    for file_path in file_list:
        if any(x in file_path.name for x in [".pak", "_pkg", "part_"]):
            continue

        raw_name = file_path.stem.replace('-', '_').replace(' ', '_')
        var_name = f"df_{raw_name}"
        single_pak = DATA_DIR / f"{raw_name}.pak"
        pkg_folder = DATA_DIR / raw_name

        if not single_pak.exists() and not pkg_folder.exists():
            df_tmp = pd.read_csv(file_path, low_memory=False)
            df_tmp.columns = [c.encode('ascii', 'ignore').decode('ascii').strip().replace(' ', '_') for c in df_tmp.columns]
            actual_size = df_tmp.memory_usage(deep=True).sum()

            if actual_size > limit_99mb:
                pkg_folder.mkdir(parents=True, exist_ok=True)
                num_splits = (actual_size // limit_99mb) + 1
                chunk_size = len(df_tmp) // num_splits

                for i, start in enumerate(range(0, len(df_tmp), chunk_size)):
                    chunk = df_tmp.iloc[start : start + chunk_size]
                    chunk.to_parquet(pkg_folder / f"{raw_name}{i}.pak", index=False)
            else:
                df_tmp.to_parquet(single_pak, index=False)

            del df_tmp
            gc.collect()

        if live_mode:
            source = pkg_folder if pkg_folder.exists() else single_pak
            if source.exists():
                globals()[var_name] = LivePacketLoader(source, var_name)

initialize_data_infrastructure(live_mode=True)

In [3]:
# stresstest
smart_df =df_sample
def verify_live_loader_integrity(smart_df):
    """
    SYSTEM-CHECK: Prüft Zahlen- und Textlogik des LivePacketLoaders.
    Entspricht deinem Standard für RAM-Schutz und saubere Tabellen.
    """
    print(f"🔍 Starte Integritäts-Check für: {smart_df.var_name}")
    print("="*50)
    results_log = {}
    try:
        mean_val = smart_df.mean()
        results_log['Mathe-Check (mean)'] = "✅ Bestanden"
    except Exception as e:
        results_log['Mathe-Check (mean)'] = f"❌ FEHLER: {e}"
    try:
        counts = smart_df['author_id'].value_counts().head(5)
        results_log['Text-Check (value_counts)'] = "✅ Bestanden"
        print("\n🏢 Top 5 Support-Accounts gefunden:")
        display(counts.to_frame())
    except Exception as e:
        results_log['Text-Check (value_counts)'] = f"❌ FEHLER: {e}"
    try:
        n_unique = smart_df['author_id'].nunique()
        results_log['Eindeutigkeit (nunique)'] = f"✅ Bestanden ({n_unique} IDs)"
    except Exception as e:
        results_log['Eindeutigkeit (nunique)'] = f"❌ FEHLER: {e}"
    try:
        preview = smart_df.head(3)
        results_log['Vorschau-Check (head)'] = "✅ Bestanden"
    except Exception as e:
        results_log['Vorschau-Check (head)'] = f"❌ FEHLER: {e}"
    print("\n" + "="*50)
    print("📊 TEST-ERGEBNISSE:")
    for test, status in results_log.items():
        print(f"{test.ljust(30)}: {status}")
    df_results = pd.DataFrame.from_dict(results_log, orient='index', columns=['Status'])
    display(df_results)
    gc.collect()

def check_loader_logic(smart_df):
    """
    Validiert die Virtual-Dataframe-Logik für Text und Zahlen.
    Erzeugt eine numerische Tabelle zur Unterstützung der Validierung.
    """
    print(f"🚀 Validierung läuft: {smart_df.var_name}")
    test_results = []
    total_len = len(smart_df)
    test_results.append({"Test": "Zeilenanzahl (Gesamt)", "Ergebnis": total_len, "Status": "✅"})
    try:
        vc = smart_df['author_id'].value_counts().head(5)
        test_results.append({"Test": "Top Author (Erster)", "Ergebnis": vc.index[0], "Status": "✅"})
        print(f"📊 Top Support-Account: {vc.index[0]} mit {vc.iloc[0]} Einträgen.")
    except Exception as e:
        test_results.append({"Test": "Text-Aggregation", "Ergebnis": str(e), "Status": "❌"})
    try:
        avg_id = smart_df['tweet_id'].mean()
        test_results.append({"Test": "Numerischer Mittelwert", "Ergebnis": f"{avg_id:.2f}", "Status": "✅"})
    except Exception as e:
        test_results.append({"Test": "Numerik-Check", "Ergebnis": "Absturz (AppleSupport-Fehler?)", "Status": "❌"})
    try:
        h = smart_df.head(2)
        test_results.append({"Test": "Head-Vorschau", "Ergebnis": f"{len(h)} Zeilen geladen", "Status": "✅"})
    except Exception as e:
        test_results.append({"Test": "Head-Check", "Ergebnis": str(e), "Status": "❌"})
    df_Check = pd.DataFrame(test_results)
    print("\n--- Zusammenfassung der Loader-Integrität ---")
    display(df_Check)

    gc.collect()

verify_live_loader_integrity(smart_df)
check_loader_logic(smart_df)

🔍 Starte Integritäts-Check für: df_sample

🏢 Top 5 Support-Accounts gefunden:


,count
author_id,
AppleSupport,13
Tesco,8
SpotifyCares,8
VirginTrains,4
105847,4



📊 TEST-ERGEBNISSE:
Mathe-Check (mean)            : ✅ Bestanden
Text-Check (value_counts)     : ✅ Bestanden
Eindeutigkeit (nunique)       : ✅ Bestanden (42 IDs)
Vorschau-Check (head)         : ✅ Bestanden


,Status
Mathe-Check (mean),✅ Bestanden
Text-Check (value_counts),✅ Bestanden
Eindeutigkeit (nunique),✅ Bestanden (42 IDs)
Vorschau-Check (head),✅ Bestanden


🚀 Validierung läuft: df_sample
📊 Top Support-Account: AppleSupport mit 13 Einträgen.

--- Zusammenfassung der Loader-Integrität ---


,Test,Ergebnis,Status
0,Zeilenanzahl (Gesamt),93,✅
1,Top Author (Erster),AppleSupport,✅
2,Numerischer Mittelwert,119285.45,✅
3,Head-Vorschau,2 Zeilen geladen,✅


In [4]:
# INFRASTRUKTUR & PFAD-MANAGEMENT & Import der in ordner enhaltenen py dateien --> darin sind alle andere importe Für das project und def
# 1. BASE_DIR: Dynamische Ermittlung des Arbeitsverzeichnisses
# stellt sicher, dass das System auch nach einem Neustart
# oder Pfadwechsel auf dem MacBook Air alles findet.
BASE_DIR = os.getcwd()

# 2. PROJECT_PATHS: Die zentrale Mapping-Tabelle (Das Skelett)
# steuert alle Ein- und Ausgabekanäle wen schon geladen werd Übersprungen
PROJECT_PATHS = {
    "REGISTRY":   os.path.join(BASE_DIR, "registry", "agents"),# YAML-Intelligence
    "DIST_CODE":  os.path.join(BASE_DIR, "output", "scripts"), # Generierte Muster
    "DIST_TEXT":  os.path.join(BASE_DIR, "output", "reports"), # Narrative Analysen
    "DIST_VIS":   os.path.join(BASE_DIR, "output", "visuals"), # Interaktive Plots
    "DATA_Sorce": os.path.join(BASE_DIR, "data")         # Deine Rohdaten
}

# 3. BEHAVIOR_DIR: Der geschützte Ort der Agenten-Rezepte
# hier liegen die YAML-Dateien, die das Verhalten steuern.
BEHAVIOR_DIR = PROJECT_PATHS["REGISTRY"]

def initialize_global_folders():
    """
    Erstellt die gesamte Projekt-Struktur auf dem MacBook.
    Stellt sicher, dass Pfade für Berichte, Scripts und Visuals existieren,
    bevor das System darauf zugreift.
    """
    # 1. Alle Pfade aus dem PROJECT_PATHS Dictionary erstellen
    for name, folder_path in PROJECT_PATHS.items():
        if not os.path.exists(folder_path):
            os.makedirs(folder_path, exist_ok=True)

    # 2. Modul-Kompatibilität für die Registry (Python-Package Standard)
    init_path = os.path.join(PROJECT_PATHS["REGISTRY"], "__init__.py")
    if not os.path.exists(init_path):
        with open(init_path, 'w') as f:
            f.write("# Agent-Registry Initialization")

initialize_global_folders()

Komponente / Funktion | Aufgabe & Operation                                                                                                                                                       | Ziel
--- |---------------------------------------------------------------------------------------------------------------------------------------------------------------------------| ---
1. Initialisierung & Deployment | Automatischer Start von Ollama und physisches Deployment der YAML-Agenten-DNA (deploy_specialist_agents_yaml) in die Registry.                                            | Sofortige Systembereitschaft und verankerte Agenten-Logik auf dem MacBook sicherstellen.
2. System-Wächter & RAM-Schutz | Überwachung der Hardware durch perform_system_check. Aktive Garbage Collection (gc.collect()), falls der RAM unter 1.5 GB fällt                                           | Systemstabilität.
3. Strategischer Veredler & RAM-Snapshot | Extraktion harter Metadaten direkt aus dem RAM (ram_facts wie dtypes, NaNs, rows). Abgleich der Benutzerfrage mit vorhandenen Spalten zur Vermeidung von Halluzinationen. | Absolute Daten-Integrität; der Agent „sieht“ die Realität des Dataframes, bevor er antwortet.
4. Task-Matrix (Routing) | Klassifizierung der Anfrage in [TASK: PLOT], [TASK: CODE] oder [TASK: EDA] basierend auf Keywords und Kontext-Wichtigkeit.                                                | Automatische Wahl des effizientesten Pfades ohne manuelles Eingreifen des Nutzers.
5. Der Neutrale Verwalter (Orchestrator) | Auswahl des passenden Spezialisten (CODE_PYTHON, TEXT_DATASCIENTIST oder PLOT_STATISTIC) über call_internal (Temperatur 0.0).                                             | Maximale Präzision durch Zuweisung der Aufgabe an den fachlich besten Experten.
6. Das Goldene Mandat (Injektion) | Kapselung von Anweisungen wie TEMP_Clear, Automated EDA und Hardware-Limits in einen unsichtbaren System-Prompt ([SYSTEM_INTERNAL]).                                      | Agenten-Eigenschaften erzwingen, ohne die Benutzer-Ausgabe mit technischem „Müll“ zu verunreinigen.
7. Spezialisierte Generierung | Pfad TEXT: Evidenz-basierte Analyse mit Zitat-Pflicht.


 Pfad CODE: Extraktion sauberer Muster aus Backticks inklusive MacBook-Air-Fix (Automatisches plt.savefig). | Fehlerfreier, direkt ausführbarer Code oder hochpräzise Berichte ohne „Ich-Form“.
8. Hygienischer Output-Filter | Harte Reinigung der Antwort (output.split(":")[-1]), um gespiegelte System-Tags oder interne Steuerbefehle zu eliminieren. | Ein sauberes, professionelles Endprodukt für die Live-Präsentation.

1. 🧹 CLEANING_PRE        → "Visualisiere ROH-Daten (vor Cleaning)"
2. 🧽 CLEANING_AGENT      → "Reinige Daten (NaN, Duplikate, Typen)"
3. 🧹 CLEANING_POST       → "Visualisiere GEREINIGTE Daten"
4. 📊 EDA_AGENT           → "Exploratory Data Analysis (Stats, Korrelationen)"
5. 📈 FEATURE_AGENT       → "Feature Engineering (neue Spalten, Encoding)"
6. 🎨 VISUALISIERUNG      → "Finale Plots/Charts/Dashboards"
7. 🤖 ML_AGENT            → "Trainiere Modelle, Evaluation, Deployment"
8. 📋 REPORTING_AGENT     → "Executive Summary + Insights"


In [ ]:
def deploy_agent_cleaning_pre():
    config = {
        "agent_name": "CLEANING_PRE",
        "role": "Data Quality Inspector (Vor Cleaning)",
        "instructions": """
        VISUALISIERE ROH-DATEN ZUSTAND:

        1. QUICK OVERVIEW:
           ```python
           print(f'Shape: {df.shape}')
           print('Datentypen:\\n', df.dtypes.value_counts())
           print('NaN pro Spalte:\\n', df.isnull().sum())
           ```

        2. 3 DIAGNOSTIK PLOTS:
           - NaN-Heatmap: sns.heatmap(df.isnull())
           - Histogram numerisch: df.hist()
           - Value-Counts Top-10 Kategorien

        SPEICHER: '01_raw_data.png'
        print('### CLEANING_PRE_DONE ###')
        """,
        "settings": {"temperature": 0.0, "max_tokens": 1500}
    }
    path = os.path.join(BEHAVIOR_DIR, "01_cleaning_pre.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_cleaning_core():
    config = {
        "agent_name": "CLEANING_CORE",
        "role": "Data Cleaning Engine",
        "instructions": """
        SYSTEMATISCHE REINIGUNG:

        1. DUPLIKATE:
           df = df.drop_duplicates()

        2. TYPEN:
           num_cols = df.select_dtypes('object').columns
           for col in num_cols:
               df[col] = pd.to_numeric(df[col], errors='coerce')

        3. NULLS:
           df[num_cols] = df[num_cols].fillna(df[num_cols].median())
           df[cat_cols] = df[cat_cols].fillna('missing')

        4. OUTLIERS (IQR):
           for col in df.select_dtypes('number'):
               Q1, Q3 = df[col].quantile([0.25, 0.75])
               df[col] = df[col].clip(Q1-1.5*(Q3-Q1), Q3+1.5*(Q3-Q1))

        VALIDATION: print(df.isnull().sum().sum())
        """,
        "settings": {"temperature": 0.0, "max_tokens": 2000}
    }
    path = os.path.join(BEHAVIOR_DIR, "02_cleaning_core.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_cleaning_post():
    config = {
        "agent_name": "CLEANING_POST",
        "role": "Quality Validator",
        "instructions": """
        POST-CLEANING VALIDATION:

        1. METRICS:
           ```python
           quality = {
               'nan_remaining': df.isnull().sum().sum(),
               'duplicates': df.duplicated().sum(),
               'memory_usage': df.memory_usage(deep=True).sum()
           }
           ```

        2. VERGLEICH PLOTS:
           - Vorher/Nachher NaN-Heatmap
           - Distributions Check

        3. QUALITY SCORE:
           score = 100 - (quality['nan_remaining']/len(df)*100)
           print(f'Data Quality: {score:.1f}%')

        SAVE: '03_cleaning_validated.png'
        """,
        "settings": {"temperature": 0.0, "max_tokens": 1500}
    }
    path = os.path.join(BEHAVIOR_DIR, "03_cleaning_post.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_eda_explorer():
    config = {
        "agent_name": "EDA_EXPLORER",
        "role": "Exploratory Data Analyst",
        "instructions": """
        KOMPLETTE EDA:

        1. DESCRIPTIVE STATS:
           display(df.describe(include='all'))

        2. KORRELATION:
           plt.figure(figsize=(12,8))
           sns.heatmap(df.corr(), annot=True, cmap='coolwarm')

        3. UNIVARIATE:
           - Num: df.hist()
           - Cat: df['col'].value_counts().plot.bar()

        4. TOP INSIGHTS:
           ```python
           insights = ['Insight 1', 'Insight 2', 'Insight 3']
           print('### KEY_FINDINGS:', insights)
           ```

        SAVE: '04_eda_complete.png'
        """,
        "settings": {"temperature": 0.1, "max_tokens": 2500}
    }
    path = os.path.join(BEHAVIOR_DIR, "04_eda_explorer.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_feature_engineer():
    config = {
        "agent_name": "FEATURE_ENGINEER",
        "role": "Feature Engineering Expert",
        "instructions": """
        INTELLIGENTE FEATURES:

        1. DATE FEATURES:
           if 'date' in df.columns:
               df['year'] = df['date'].dt.year
               df['month'] = df['date'].dt.month

        2. TEXT FEATURES:
           if 'text' in df.columns:
               df['text_len'] = df['text'].str.len()
               df['word_count'] = df['text'].str.split().str.len()

        3. INTERACTIONS:
           df['col1_x_col2'] = df['col1'] * df['col2']

        4. BINNING:
           df['age_group'] = pd.cut(df['age'], bins=5)

        FEATURE MATRIX:
           print('New Features:', df.columns[-5:].tolist())
        """,
        "settings": {"temperature": 0.0, "max_tokens": 2500}
    }
    path = os.path.join(BEHAVIOR_DIR, "05_feature_engineer.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_visual_master():
    config = {
        "agent_name": "VISUAL_MASTER",
        "role": "Visualization Architect",
        "instructions": """
        PROFESSIONELLE VISUALS:

        1. MULTIVARIATE:
           sns.pairplot(df, hue='target')

        2. FACETGRID:
           g = sns.FacetGrid(df, col='category', col_wrap=3)
           g.map(sns.histplot, 'value')

        3. INTERACTIVE:
           ```python
           import plotly.express as px
           fig = px.scatter(df, x='x', y='y', color='cat')
           fig.write_html('dashboard.html')
           ```

        4. SUBPLOTS:
           fig, axes = plt.subplots(2, 2, figsize=(15,10))
           # 4 verschiedene Views
        """,
        "settings": {"temperature": 0.1, "max_tokens": 3000}
    }
    path = os.path.join(BEHAVIOR_DIR, "06_visual_master.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_ml_pipeline():
    config = {
        "agent_name": "ML_PIPELINE",
        "role": "Machine Learning Pipeline Engineer",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        KOMPLETTE ML PIPELINE (Production-Ready):

        1. TRAIN/TEST SPLIT:
           ```python
           from sklearn.model_selection import train_test_split
           X = df.drop('target', axis=1) if 'target' in df else df.iloc[:,:-1]
           y = df['target'] if 'target' in df else df.iloc[:,-1]
           X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
           ```

        2. PREPROCESSING PIPELINE:
           ```python
           from sklearn.pipeline import Pipeline
           from sklearn.preprocessing import StandardScaler, OneHotEncoder
           from sklearn.compose import ColumnTransformer

           numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
           categorical_features = X.select_dtypes(include=['object']).columns

           preprocessor = ColumnTransformer(
               transformers=[
                   ('num', StandardScaler(), numeric_features),
                   ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
               ])
           ```

        3. BASELINE MODELLE:
           ```python
           from sklearn.ensemble import RandomForestClassifier
           from sklearn.linear_model import LogisticRegression

           rf = Pipeline([('preprocessor', preprocessor),
                         ('classifier', RandomForestClassifier(n_estimators=100))])
           rf.fit(X_train, y_train)
           print(f"RF Accuracy: {rf.score(X_test, y_test):.3f}")
           ```

        SAVE: joblib.dump(rf, 'ml_baseline.pkl')
        """,
        "settings": {"temperature": 0.0, "max_tokens": 3500}
    }
    path = os.path.join(BEHAVIOR_DIR, "07_ml_pipeline.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_model_evaluator():
    config = {
        "agent_name": "MODEL_EVALUATOR",
        "role": "Model Performance Analyst",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        UMSTANDENDE MODELLEVALUATION:

        1. CROSS-VALIDATION:
           ```python
           from sklearn.model_selection import cross_val_score
           scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
           print(f"CV Scores: {scores.mean():.3f} ± {scores.std():.3f}")
           ```

        2. DETAILED METRICS:
           ```python
           from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
           y_pred = model.predict(X_test)
           print(classification_report(y_test, y_pred))
           print("Confusion Matrix:\\n", confusion_matrix(y_test, y_pred))
           ```

        3. LEARNING CURVES:
           ```python
           from sklearn.model_selection import learning_curve
           train_sizes, train_scores, val_scores = learning_curve(
               model, X_train, y_train, cv=5)
           plt.plot(train_sizes, val_scores.mean(axis=1), label='Validation')
           plt.savefig('08_learning_curves.png')
           ```

        4. PERFORMANCE TABLE:
           ```python
           results = pd.DataFrame({
               'Metric': ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'],
               'Score': [0.85, 0.84, 0.87, 0.85, 0.89]
           })
           print(results)
           """,
        "settings": {"temperature": 0.0, "max_tokens": 2500}
    }
    path = os.path.join(BEHAVIOR_DIR, "08_model_evaluator.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_hyper_optimizer():
    config = {
        "agent_name": "HYPER_OPTIMIZER",
        "role": "Hyperparameter Tuning Expert",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        AUTOMATISCHE HYPERPARAMETER OPTIMIERUNG:

        1. OPTUNA STUDY:
           ```python
           import optuna
           def objective(trial):
               params = {
                   'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                   'max_depth': trial.suggest_int('max_depth', 3, 10),
                   'min_samples_split': trial.suggest_int('min_samples_split', 2, 20)
               }
               model = RandomForestClassifier(**params)
               model.fit(X_train, y_train)
               return model.score(X_test, y_test)

           study = optuna.create_study(direction='maximize')
           study.optimize(objective, n_trials=50)
           ```

        2. BESTE PARAMETER:
           ```python
           print("Beste Parameter:", study.best_params)
           print("Bester Score:", study.best_value)

           best_model = RandomForestClassifier(**study.best_params)
           best_model.fit(X_train, y_train)
           joblib.dump(best_model, '09_optimized_model.pkl')
           ```

        3. PARAMETER VISUALIZATION:
           ```python
           optuna.visualization.plot_optimization_history(study)
           optuna.visualization.plot_param_importances(study)
           """,
        "settings": {"temperature": 0.0, "max_tokens": 3000}
    }
    path = os.path.join(BEHAVIOR_DIR, "09_hyper_optimizer.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_reporting_exec():
    config = {
        "agent_name": "REPORTING_EXEC",
        "role": "Executive Data Scientist",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        EXECUTIVE SUMMARY REPORT (Markdown):

        1. PROBLEM & RESULT:
           ```markdown
           # 🎯 DATA SCIENCE REPORT

           ## Problem
           [1-Satz Problemstellung aus Daten]

           ## Ergebnisse
           - Accuracy: 87.3% (Test Set)
           - Top Feature: [Spalte] (23% Importance)
           - Business Impact: +15% Umsatzprognose
           ```

        2. VISUAL HIGHLIGHTS:
           ```markdown
           ## 📊 Key Visuals
           ![Korrelationsmatrix](04_eda_correlation.png)
           ![Feature Importance](feature_importance.png)
           ![Learning Curve](08_learning_curves.png)
           ```

        3. ACTION ITEMS:
           ```markdown
           ## ✅ NÄCHSTE SCHRITTE
           1. [Modell in Prod] - Data Engineering Team - 2 Wochen
           2. [A/B Test starten] - Product Team - 4 Wochen
           3. [Monitoring Setup] - ML Ops - 1 Woche
           ```

        SAVE: executive_report.md
        """,
        "settings": {"temperature": 0.1, "max_tokens": 2000}
    }
    path = os.path.join(BEHAVIOR_DIR, "10_reporting_exec.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_code_python():
    config = {
        "agent_name": "CODE_PYTHON",
        "role": "Senior Python Architect (Data Engineering)",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        DU BIST EIN DATEN-ENGINEER. DEINE DNA ERZWINGT FEHLERFREIEN CODE:

        1. LADE-VERBOT: Nutze NIEMALS pd.read_csv(). Variable 'df' ist im RAM.
        2. DATA-INTEGRITY: df.dtypes prüfen, pd.to_numeric() falls nötig
        3. LOGIK: Komplexe Filter, Groupbys, Features erstellen
        4. FINAL: df_ML = df.copy() falls gewünscht

        ANTWORTE NUR MIT CODE IN ```python``` BACKTICKS. KEINE PROSA.
        """,
        "settings": {"temperature": 0.0, "max_tokens": 2500}
    }
    path = os.path.join(BEHAVIOR_DIR, "11_code_python.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_text_datascientist():
    config = {
        "agent_name": "TEXT_DATASCIENTIST",
        "role": "Senior Data Scientist (Evidence Notary)",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        DATENANALYSE (Sauber & Evidenzbasiert):

        1. DATEN-PROTOKOLL:
           | Spalte | Datentyp | NaN | Unique | Kardinalität |
           |--------|----------|-----|--------|--------------|

        2. STRATEGISCHE EINSCHÄTZUNG:
           - Key Features vs. Noise Spalten
           - Datenqualität (NaN, Skewness, etc.)
           - ML-Vorbereitung (TEMP_Clear)

        3. TOOL AUDIT:
           - ydata_profiling, sweetviz verfügbar?
           - pip install [Package] falls nötig

        Beende mit: ### AGENT_PROCESS_COMPLETE ###
        """,
        "settings": {"temperature": 0.0, "max_tokens": 3000}
    }
    path = os.path.join(BEHAVIOR_DIR, "12_text_datascientist.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)


def deploy_complete_pipeline():
    """🚀 DEPLOY ALLE 12 DATA SCIENCE AGENTEN"""
    agents = [
        deploy_agent_cleaning_pre,
        deploy_agent_cleaning_core,
        deploy_agent_cleaning_post,
        deploy_agent_eda_explorer,
        deploy_agent_feature_engineer,
        deploy_agent_visual_master,
        deploy_agent_ml_pipeline,
        deploy_agent_model_evaluator,
        deploy_agent_hyper_optimizer,
        deploy_agent_reporting_exec,
        deploy_agent_code_python,
        deploy_agent_text_datascientist
    ]

    for agent in agents:
        agent()
    #print("📁 Registry:", BEHAVIOR_DIR)
deploy_complete_pipeline()

In [ ]:
# AGENTEN-DEPLOYMENT (ERWEITERTE YAML-REGISTRY)

def deploy_agent_code_python():
    """
    DNA: SENIOR PYTHON DEVELOPER (AUTONOM)
    Ziel: Komplexe Daten-Transformationen & Berechnungen.
    """
    config = {
        "agent_name": "CODE_PYTHON",
        "role": "Senior Python Architect (Data Engineering)",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "heartbeat_signal": "### PHASE:STEP_SUCCESSFUL ###",
        "instructions": """
        DU BIST EIN DATEN-ENGINEER. DEINE DNA ERZWINGT FEHLERFREIEN CODE:

        1. LADE-VERBOT: Nutze NIEMALS pd.read_csv(). Variable 'df' ist im RAM.
        2. DATA-INTEGRITY:
           - Prüfe Typen mit df.dtypes.
           - Wandle falls nötig: pd.to_numeric(df[col], errors='coerce').
           - print("### PHASE:DATA_VALIDATION_READY ###")  <-- TIMER RESET
        3. LOGIK-PHASE:
           - Erstelle komplexe Filter, Groupbys oder neue Features.
           - Nutze immer 'df' als Basis für Transformationen.
           - print("### PHASE:LOGIC_CALC_READY ###")       <-- TIMER RESET
        4. FINALISIERUNG:
           - Überschreibe das globale df_ML falls gewünscht: df_ML = df.copy()
           - print("### AGENT_PROCESS_COMPLETE ###")

        ANTWORTE NUR MIT CODE IN BACKTICKS. KEINE PROSA.
        """,
        "settings": {"temperature": 0.0, "max_tokens": 2500, "timeout_window": 1200}
    }
    path = os.path.join(BEHAVIOR_DIR, "code_python.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_text_datascientist():
    """
    DNA: EVIDENZ-NOTAR & ANALYST (V16.0 - Enterprise Hybrid)
    Ziel: Hochpräzise, daten-zitierende Dokumentation mit ReAct-Zyklus & Tool-Audit.
    """

    config = {
        "agent_name": "TEXT_DATASCIENTIST",
        "role": "Senior Data Scientist (Technical Evidence Notary)",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        ### STRIKTESTE VERBOTE - BEI VERSTOSS DROHT SYSTEM-ABBRUCH:
        - KEIN "ICH", "ALS", "MEIN", "WIR", "UNSER".
        - KEINE EINLEITUNG. STARTE DIREKT MIT # DATEN-PROTOKOLL.
        - KEIN STOTTERN: WIEDERHOLE NIEMALS ZAHLEN ODER TYPEN AUS DER TABELLE IM TEXT.
        - KEIN SMALLTALK, KEINE HÖFLICHKEIT, KEINE BESCHREIBUNG DEINER ROLLE.

        ### LOGISCHER ABLAUF (REACT-DNA):
        1. THOUGHT: Kurze interne Analyse der Datei [df_name] und des [FAKTEN-KONTEXT].
        2. STATUS-SIGNAL: Melde sofort: ### PHASE: THINKING_READY ###

        3. ÜBERSCHRIFT: # DATEN-PROTOKOLL: [df_name]
        4. BEWEISAUFNAHME (Tabelle): Erstelle die Tabelle in pandas format df_info mit folgenden spalten: | Spalte | Datentyp | NaN | Unique | Kardinalität % |
        5. TEMP_Clear (ML-Veredelung): Fordere explizite Korrekturen (Casts, Null-Handling) für df_ML.
        6. ANALYSE: Strategische Einordnung basierend auf harten Fakten ohne Zahlen-Wiederholung.
        7. UPGRADE & TOOL-AUDIT (Projekt-Spezifisch):
           - Prüfe auf Verfügbarkeit von:
             a) EDA-Automatisierung: ydata_profiling, sweetviz, autoviz.
             b) ML-Vorbereitung: scikit-learn (für Skalierung/Encoding).
             c) Image-Processing: TFRecord/image package.
           - Falls diese fehlen und für den [df_name] kritisch sind (z.B. TFRecord bei Bildern), nenne den exakten Installationsbefehl.
        DU BIST EIN TECHNISCHER NOTAR DER DATEN. NUTZE DEN REACT-ZYKLUS (THOUGHT -> ACTION -> OBSERVATION).
        DEINE ANTWORT IST REIN EVIDENZBASIERT UND DEINE DNA ERZWINGT FOLGENDE LOGISCHE REIHENFOLGE:

        ### DETAIL-INSTRUKTIONEN:
        - PERSPEKTIVE: Handle als Sprachrohr des Dataframes. "Die Datenlage in [df_name] belegt...".
        - ZITAT-PFLICHT: Beziehe dich bei jeder Analyse-Aussage direkt auf Spaltennamen aus dem [FAKTEN-KONTEXT].
        - TOOL-AUDIT: Prüfe aktiv auf verfügbare automatisierte EDA-Tools.
        - OBJEKTIVITÄT: Nutze Fachbegriffe (Kardinalität, Saisonalität, Skewness). Keine Vermutungen.

        ### FINALE SIGNALE:
        - Beende JEDE Antwort mit dem exakten Signal: ### AGENT_PROCESS_COMPLETE ###

        STRIKTE HANDSCHELLEN (RECAP):
        - Informationen aus der Tabelle dürfen NICHT im Fließtext vorkommen.
        - Keine Sätze wie "As a Senior Data Scientist" oder "Ich habe analysiert".
        - Kein Python-Code im Output (außer pip install Befehle im Upgrade-Bereich).

        1. PERSPEKTIVE & TOOL-AUDIT (Schritt 1):
           - THOUGHT: Welche Werkzeuge stehen mir zur Verfügung (Squeezer, Kardinalitäts-Check)?
           - STARTE direkt mit der Überschrift: # DATEN-PROTOKOLL: [df_name]
           - [TOOL_CHECK]: Prüfe auf ydata_profiling, sweetviz, sklearn. Falls nicht geladen, am Ende Installation vorschlagen.
           - Handle als Sprachrohr des Dataframes: Ersetze "Ich sehe" durch "Die Datenlage in [df_name] belegt...".
           - Bestätige sofort die Dimensionen (Zeilen x Spalten) und melde: ### PHASE: THINKING_READY ###

        2. STRUKTURELLE EVIDENZ (Tabelle - Schritt 2):
           - Erstelle IMMER eine df_info Tabelle: | Spalte | Datentyp | Fehlende Werte | Eindeutige Werte | Kardinalität % |
           - Dies ist die fundamentale Beweisaufnahme. Alle numerischen Metadaten gehören NUR hierhin.
           - MERKMAL-ANALYSE: Bewerte die Spalten nach strategischer Relevanz (Key-Features vs. Noise).
           - TEMP_Clear (ML-Vorbereitung): Fordere eindeutige Korrekturen (z.B. Datentyp-Casts oder Null-Wert-Checks).

        3. ANALYSE-PHASE (Strategische Einordnung - Schritt 3):
           - Nutze Fachterminologie: Kardinalität, Saisonalität, Füllrate, Skewness.
           - AUTOMATED EDA: Fasse Trends und Auffälligkeiten basierend auf harten Fakten zusammen.
           - Strikte Objektivität: Keine Vermutungen, kein "wahrscheinlich". Zitiere exakte Werte aus dem [FAKTEN-KONTEXT].

        4. SYSTEM-UPGRADE & SIGNALE (Schritt 4):
           - Falls EDA-Werkzeuge fehlen: Benenne am Ende explizit die nötigen Pakete für tiefere Filterung (z.B. pip install ydata-profiling).
           - Beende IMMER mit dem Signal: ### AGENT_PROCESS_COMPLETE ###

        STRIKTE VERBOTE (Die Handschellen):
        - STOTTER-VERBOT: Informationen aus der Tabelle (Null-Werte, Typen) NIEMALS im Fließtext wiederholen.
        - IDENTITÄTS-VERBOT: KEINE Sätze wie "As a Senior Data Scientist" oder "Ich habe analysiert".
        - HYGIENE-VERBOT: KEINE Höflichkeitsfloskeln, KEIN Python-Code, KEIN Smalltalk.
        - KEINE Einleitungssätze über deine eigene Funktion oder Rolle.

        """,
        "settings": {
            "temperature": 0.0,
            "language": "de",
            "max_tokens": 3000,
            "timeout_window": 1200
        }
    }

    # Sicherstellen, dass das Verzeichnis existiert
    os.makedirs(BEHAVIOR_DIR, exist_ok=True)
    path = os.path.join(BEHAVIOR_DIR, "text_datascientist.yaml")

    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_plot_statistic():
    """
    DNA: STATISTIK-PROFI & VISUALIZER (MAX-PERFEKTION)
    Ziel: Autonome Erstellung semantischer Plots mit Artefakt-Reporting.
    """
    config = {
        "agent_name": "PLOT_STATISTIC",
        "role": "Senior Statistics Expert (Visual Intelligence)",
        "output_type": "visual",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        DU BIST EIN VISUALISIERUNGS-PROFI. DEINE DNA ERZWINGT TOTAL-AUTONOMIE:

        1. TITEL-PHASE:
           - Analysiere die text_user und erstelle einen kurzen Titel (Slug).
           - Definiere: p_name = "plot_" + titel.replace(" ", "_").lower() + ".html"
           - print("### PHASE:NAMING_READY ###")            <-- TIMER RESET

        2. VISUAL-PHASE:
           - Erstelle 'fig' mit Plotly Express (px). Nutze passende Farbskalen.
           - fig.update_layout(title=titel, template='plotly_dark')
           - fig.show()
           - fig.write_html(p_name)
           - print("### PHASE:PLOT_GENERATION_READY ###")   <-- TIMER RESET

        3. DATA-PHASE (Validierung):
           - Erstelle plot_df = df.head(15).
           - display(plot_df)
           - print("### PHASE:DATA_VALIDATION_READY ###")   <-- TIMER RESET

        4. ARTEFAKT-REPORT & SHUTDOWN:
           - print(f"ANALYSE: Erkläre kurz den Plot auf Deutsch.")
           - print(f"### ARTIFACT:{p_name}")                 <-- ORCHESTRATOR SIGNAL
           - print("### AGENT_PROCESS_COMPLETE ###")        <-- FINAL STOP

        RECHTE: Du darfst os und shutil nutzen. LADE-VERBOT: Kein pd.read_csv!
        """,
        "settings": {"temperature": 0.0, "max_tokens": 2500, "timeout_window": 1200}
    }
    path = os.path.join(BEHAVIOR_DIR, "plot_statistic.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_specialist_agents_yaml():
    """
    SYSTEM-ARCHITEKT (V5.1): Koordiniert das Deployment aller Agenten.
    Verankert Rollen, Hardware-Limits und Sicherheit-Einstellungen
    dauerhaft in der Registry (BEHAVIOR_DIR).
    """
    if not os.path.exists(BEHAVIOR_DIR):
        os.makedirs(BEHAVIOR_DIR, exist_ok=True)
    try:
        deploy_agent_code_python()
        deploy_agent_text_datascientist()
        deploy_agent_plot_statistic()

    except Exception as e:
        print(f"❌ Fehler bei Deployment: {e}")
deploy_specialist_agents_yaml()


In [5]:
# LLM-AGENTEN-STEUERUNG Anpassung benutzereingabe weiterleitung llm

# 1. VERBINDUNGS-CHECK (Auto-Check/Start)
def pruefe_und_starte_ollama():
    url = "http://localhost:11434"
    try:
        if requests.get(url, timeout=1).status_code == 200: return True
    except: pass
    try:
        if sys.platform == "darwin":
            subprocess.Popen(["open", "-a", "Ollama"])
        elif sys.platform == "win32":
            p = os.path.expandvars(r"%LocalAppData%\Ollama\ollama app.exe")
            subprocess.Popen([p] if os.path.exists(p) else ["ollama", "serve"])
            if os.path.exists(p):
                subprocess.Popen([p])
            else:
                subprocess.Popen(["ollama", "serve"], shell=True)
        for _ in range(10):
            time.sleep(1)
            try:
                if requests.get(url, timeout=1).status_code == 200:
                    return True
            except: continue
    except Exception as e:
        print(f"⚠️ Fehler beim Startversuch: {e}")
    print("\n🛑 START FEHLGESCHLAGEN. Nächste Schritte:")
    print("1. Installieren (ollama.com)")
    return False

# 2. INFO-RUN_RAM_CPU_CHECK
def start_spinner(stop_event, is_de=True):
    """
    VISUELLES FEEDBACK [UX-STANDARD]
    Zeigt einen Spinner während der LLM-Verarbeitung.
    """
    def perform_system_check():
        """
        SYSTEM-CHECK: Validiert RAM & CPU-Last.
        Hält 1 Core frei.
        """
        ram_avail = psutil.virtual_memory().available / (1024**3)
        cpu_usage = psutil.cpu_percent(interval=None) # Hier wurde die Definition gefehlt
        status_msg = ""
        if ram_avail < 1.5:
            gc.collect()
            status_msg = f"⚠️ RAM kritisch ({ram_avail:.2f}GB). GC ausgeführt."
        if cpu_usage > 90:
            active_cores = os.cpu_count() - 1
            cpu_alert = f" | 🔥 CPU Last hoch ({cpu_usage}%)."
            status_msg = status_msg + cpu_alert if status_msg else cpu_alert
        return status_msg

    chars = ['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏']
    msg = "  🔄 Analysiere (Lokale GPU/CPU)..." if is_de else "  🔄 Analyzing (Local GPU/CPU)..."
    idx = 0
    while not stop_event.is_set():
        alert_msg = perform_system_check()
        sys.stdout.write(f'\r{msg} {chars[idx % len(chars)]} {alert_msg}')
        sys.stdout.flush()
        idx += 1
        time.sleep(0.2)
    sys.stdout.write('\r' + ' ' * 100 + '\r')
    sys.stdout.flush()

def check_available_tools():
    """
    Zentrale Tool-Prüfung & Wissens-Bibliothek (V4.1) - Enterprise AI Stack.
    Liefert Status + Logik-Zuordnung für den Epic-Richter.
    Basiert auf dem 'Warenordnung & Verständnis'-Prinzip.
    """
    import importlib.util
    import json
    import os
    import networkx as nx
    auto_tools = {

        # --- CORE LOGIC & SYSTEM (Wichtig für MB-Fragen & OS-Zugriff) ---
        'Python_OS':    {'pkg': 'os',           'phase': 'SYSTEM', 'priority': 0},
        'Pandas_Core':  {'pkg': 'pandas',       'phase': 'DATA',   'priority': 0},
        'System_Stats': {'pkg': 'psutil',       'phase': 'SYSTEM', 'priority': 0},


        # Diese Tools retten die Reihenfolge und verstehen den Intent
        'SpaCy':        {'pkg': 'spacy',        'phase': 'LOGIC_NLP',  'priority': 0},
        'NetworkX':     {'pkg': 'networkx',     'phase': 'LOGIC_GRAPH','priority': 0},
        'LlamaIndex':   {'pkg': 'llama_index',  'phase': 'LOGIC_META', 'priority': 0},

        # --- REINIGUNG & VALIDIERUNG (Priority 1) ---
        'Klib':               {'pkg': 'klib',               'phase': 'CLEAN', 'priority': 1},
        'Great-Expectations': {'pkg': 'great_expectations', 'phase': 'QA',    'priority': 1},

        # --- EDA & VISUALISIERUNG (Phase: Analyse) ---
        'AutoViz':         {'pkg': 'autoviz',         'phase': 'EDA', 'priority': 2},
        'ydata_profiling': {'pkg': 'ydata_profiling', 'phase': 'EDA', 'priority': 2},
        'Sweetviz':        {'pkg': 'sweetviz',        'phase': 'EDA', 'priority': 2},
        'D-Tale':          {'pkg': 'dtale',           'phase': 'EDA', 'priority': 2},
        'Klib':            {'pkg': 'klib',            'phase': 'CLEAN', 'priority': 1},
        'Lux':             {'pkg': 'lux',             'phase': 'EDA', 'priority': 2},
        'MLBox_Prep':   {'pkg': 'mlbox',         'phase': 'PREP',     'priority': 2},

        # --- AUTO-ML (Phase: Modellierung) ---
        'H2O_ai':       {'pkg': 'h2o',           'phase': 'ML',       'priority': 3},
        'TPOT':         {'pkg': 'tpot',          'phase': 'ML',       'priority': 3},
        'PyCaret':      {'pkg': 'pycaret',       'phase': 'ML',       'priority': 3},
        'Auto-Sklearn': {'pkg': 'autosklearn',   'phase': 'ML',       'priority': 3},
        'MLBox':        {'pkg': 'mlbox',         'phase': 'PREP',     'priority': 2},

        # --- VALIDIERUNG & FORECASTING ---
        'Great-Expectations': {'pkg': 'great_expectations', 'phase': 'QA', 'priority': 1},
        'Prophet':            {'pkg': 'prophet',            'phase': 'FORECAST', 'priority': 3},
        'NeuralProphet':      {'pkg': 'neuralprophet',      'phase': 'FORECAST', 'priority': 3},

        # --- LLM & INTERAKTION (Phase: Interface) ---
        'PandasAI':  {'pkg': 'pandasai',  'phase': 'CHAT', 'priority': 4},
        'LangChain': {'pkg': 'langchain', 'phase': 'AGENT', 'priority': 4},
        'Giskard':   {'pkg': 'giskard',   'phase': 'QA',    'priority': 5},

        # --- VALIDIERUNG & FORECASTING ---
        'Great-Expectations': {'pkg': 'great_expectations', 'phase': 'QA', 'priority': 1},
        'Prophet':            {'pkg': 'prophet',            'phase': 'FORECAST', 'priority': 3},
        'NeuralProphet':      {'pkg': 'neuralprophet',      'phase': 'FORECAST', 'priority': 3},

        # --- ERKLÄRBARKEIT (XAI) ---
        'SHAP':  {'pkg': 'shap',  'phase': 'XAI', 'priority': 4},
        'LIME':  {'pkg': 'lime',  'phase': 'XAI', 'priority': 4},
        'Dalex': {'pkg': 'dalex', 'phase': 'XAI', 'priority': 4},

        # --- TEXT & SENTIMENT ---
        'Transformers': {'pkg': 'transformers', 'phase': 'NLP', 'priority': 3},
        'SpaCy':        {'pkg': 'spacy',        'phase': 'NLP', 'priority': 3},
        'TextBlob':     {'pkg': 'textblob',     'phase': 'NLP', 'priority': 3}
    }
    tools_status = {}
    logic_map = nx.DiGraph()
    for tool_name, info in auto_tools.items():
        try:
            spec = importlib.util.find_spec(info['pkg'])
            is_installed = spec is not None
            tools_status[tool_name] = {
                'installed': is_installed,
                'phase': info['phase'],
                'priority': info['priority'],
                'package': info['pkg']
            }
            if is_installed:
                logic_map.add_node(tool_name, phase=info['phase'], priority=info['priority'])
        except:
            tools_status[tool_name] = {
                'installed': False,
                'phase': info['phase'],
                'priority': info['priority'],
                'package': info['pkg']
            }
    tools_status['__LOGIC_CORE__'] = {
        'intent_engine': 'SpaCy-Ready',
        'knowledge_graph': list(logic_map.nodes()),
        'priority_flow': sorted(
            [(t, info['priority']) for t, info in auto_tools.items()],
            key=lambda x: x[1]
        ),
        'total_tools_monitored': len(auto_tools)
    }
    registry_path = "tool_logic_registry.json"
    with open(registry_path, 'w', encoding='utf-8') as f:
        json.dump(tools_status, f, indent=4)
    return tools_status


# 3.LLM Arbeits blöcke für def_frage
# INTUITIONS-CHECK & VEREDELUNG
def veredle_input(frage):
    """
    MODULARE LOGIK: Überbrückt die Lücke zwischen Intuition und Technik.
    Vorteil: Erkennt Absichten (Stimmung/Zeit), auch wenn Namen fehlen.
    """
    f_up = frage.upper()
    veredelt = frage

    # Konzeptionelle Verknüpfung (Vermeidet 'Blinde Absichten')
    if "STIMMUNG" in f_up or "SENTIMENT" in f_up:
        veredelt += " (Nutze bevorzugt die Spalte 'sentiment' für die Analyse)."

    if "ZEIT" in f_up or "VERLAUF" in f_up:
        veredelt += " (Prüfe die Spalte 'date' für Zeitreihen-Plots)."

    return veredelt

# FINDE DATEN DIE ZUR FRAGE PASSEN QUELLE
def hole_aktiven_dataframe_namen(suchbegriff=None):
    """
    UNIVERSAL-DETEKTIV:
    - FIX: Loader/Packets werden NICHT mehr übersprungen (Typ-Priorität vor Größe).
    - FIX: Behandelt NaN/None/Inf tolerant (Score-System statt Filter-Sperre).
    - Volle Proxy/Packet-Erkennung [2026-01-07].
    - Beweis-Kontext für SYSTEM-SNAPSHOTS.
    """
    import pandas as pd
    import numpy as np
    import difflib
    from IPython import get_ipython
    from pathlib import Path

    shell = get_ipython()
    ns = shell.user_ns if shell else globals()

    def get_priority_score(name):
        obj = ns[name]
        score = 0
        if hasattr(obj, 'get_packet') or 'Loader' in type(obj).__name__:
            score += 5000
        elif isinstance(obj, pd.DataFrame):
            score += 1000
        else:
            return 0
        if "TEMP" in name.upper(): score += 500
        try:
            if hasattr(obj, 'row_count'): s = obj.row_count
            elif hasattr(obj, 'shape'): s = obj.shape[0]
            else: s = len(obj)

            val = pd.to_numeric(s, errors='coerce')
            if np.isfinite(val): score += int(val)
        except:
            pass
        return score
    alle_namen = [n for n in ns if not n.startswith('_') and n != 'BEWEIS_KONTEXT']
    daten_namen = [n for n in alle_namen if get_priority_score(n) > 0]

    if not daten_namen:
        if shell:
            for cell in reversed(ns.get('In', [])[-15:]):
                if '=' in cell and any(kw in cell for kw in ['pd.', 'Loader', 'read_', 'Packet']):
                    potenzial = cell.split('=')[0].strip().split()[-1]
                    if potenzial in ns:
                        daten_namen = [potenzial]
                        break
        if not daten_namen: return "Unbekannter_Datensatz"
    allgemeine_begriffe = ["datensatz", "daten", "tabelle", "übersicht", "alle"]
    f_search = suchbegriff.lower() if suchbegriff else ""
    ist_allgemeine_anfrage = any(w in f_search for w in allgemeine_begriffe)
    if "df_TEMP" in daten_namen and not ist_allgemeine_anfrage:
        bester_name = "df_TEMP"
    else:
        bester_name = max(daten_namen, key=get_priority_score)
    obj_ref = ns[bester_name]
    spalten = []
    if hasattr(obj_ref, 'columns'): spalten = list(obj_ref.columns)
    elif hasattr(obj_ref, 'get_schema'):
        res = obj_ref.get_schema()
        spalten = res.get('columns', []) if isinstance(res, dict) else []
    row_count = "Live/Initialisierung"
    try:
        if hasattr(obj_ref, 'shape'): row_count = obj_ref.shape[0]
        elif hasattr(obj_ref, 'row_count'): row_count = obj_ref.row_count
    except: pass
    ns['BEWEIS_KONTEXT'] = {
        "fokus_quelle": bester_name,
        "dimensionen": f"{row_count} Zeilen x {len(spalten)} Spalten",
        "merkmale": spalten,
        "objekt_typ": type(obj_ref).__name__
    }
    if ist_allgemeine_anfrage and len(daten_namen) > 1:
        return [n for n in daten_namen if "TEMP" not in n.upper()]
    return bester_name

# FINDE IN DEN GEFUNDENEN
def extrahiere_ram_fakten(df_name):
    """
    INDEX-FILTER & BEWEIS-EXTRAKTOR 2.0:
    Unterstützt pd.DataFrame UND LivePacketLoader-Proxies
    Identifiziert Klassen, Zeitspannen
    """
    import pandas as pd
    source = globals().get(df_name)
    if source is None:
        return {"df_name": "Unbekannt", "rows": 0, "cols": [], "error": "Quelle nicht gefunden"}
    if not isinstance(source, pd.DataFrame) and hasattr(source, 'get_schema'):
        schema = source.get_schema()
        return {
            "df_name": df_name,
            "rows": schema.get('row_count', 'unbekannt'),
            "cols": schema.get('columns', []),
            "dtypes": schema.get('types', {}),
            "source_type": "Proxy/Packet (Lazy Loading)"
        }
    facts = {
        "df_name": df_name,
        "rows": len(source),
        "cols": list(source.columns),
        "dtypes": source.dtypes.astype(str).to_dict(),
        "memory_mb": round(source.memory_usage(deep=True).sum() / 1024**2, 2),
        "source_type": "pd.DataFrame"
    }
    date_cols = source.select_dtypes(include=['datetime64', 'timedelta64']).columns
    if not date_cols.empty:
        facts["time_range"] = {
            str(col): f"{source[col].min()} bis {source[col].max()}"
            for col in date_cols
        }
    klassen_beweise = {}
    obj_cols = source.select_dtypes(include=['object', 'category']).columns
    for col in obj_cols:
        uniques = source[col].unique()
        count_uniques = len(uniques)
        if count_uniques < 50:
            klassen_beweise[col] = {
                "anzahl_klassen": count_uniques,
                "beispiele": [str(x) for x in uniques[:15]],
                "status": "KLASSE ERKANNT"
            }
    nans = source.isnull().sum()
    facts["nans"] = nans[nans > 0].to_dict()
    facts["klassen_analyse"] = klassen_beweise
    facts["data_preview"] = source.head(3).to_dict()
    return facts

# Strategische weiche Spezifiziere um zeit zu sparen
def bestimme_task_prefix(text_upper):
    """
    STRATEGISCHE SCHALTZENTRALE:
    Ordnet der Anfrage ein Label zu, damit der Verwalter den richtigen Agenten wählt.
    Priorität: PLOT > CODE > EDA > UNKNOW
    """
    if any(k in text_upper for k in ["PLOT", "GRAFIK", "CHART", "VISUALISIER", "DIAGRAMM"]):
        return "[TASK: PLOT]"
    if any(k in text_upper for k in ["CODE", "MUSTER", "BERECHN", "REINIG", "TRANSFORM", "FIX", "KORRIGIER"]):
        return "[TASK: CODE]"
    if any(k in text_upper for k in ["WAS", "WARUM", "ANALYSIER", "BESCHREIB", "ZUSAMMENFASS", "STATISTIK"]):
        return "[TASK: EDA]"
    if any(k in text_upper for k in ["SUCHE", "FINDE", "HTTP", "WWW", "INTERNET", "HALLO", "WER", "WIE"]):
        return "[TASK: UNKNOW]"
    return "[TASK: ANALYSIS]"

# verbindet die anfrage vom benutzer mit der abgleich der daten zur feinjustage der anfrage
def erstelle_optimierte_anfrage(text_user, ram_facts):
    """
    AUFTRAGS-VEREDELUNG:
    Veredelt die Intuition (User) und injiziert die Beweise als Dossier.
    """
    veredelt = veredle_input(text_user)
    v_low = veredelt.lower()
    f_up = veredelt.upper()
    prefix = bestimme_task_prefix(f_up)
    df_n  = ram_facts.get("df_name", "Unbekannter_Datensatz")
    cols  = ram_facts.get("cols", [])
    rows  = ram_facts.get("rows", "unbekannt")
    stats = ram_facts.get("stats_sample", {})
    gefundene = [s for s in cols if s.lower() in v_low]
    warnung = ""
    wichtige_begriffe = [w for w in v_low.split() if w.isdigit() and len(w) == 4]
    for b in wichtige_begriffe:
        if not any(b in str(v) for v in cols) and not any(b in str(v) for v in stats.values()):
            warnung += f" ⚠️ HINWEIS: '{b}' nicht in {df_n} gefunden!"
    context = f"\n- FOKUS-SPALTEN: {gefundene}" if gefundene else ""
    dossier = f"""
{prefix}
### SYSTEM-SNAPSHOT (ram_facts):
- AKTIVER DATENSATZ: {df_n}
- DIMENSIONEN: {rows} Zeilen x {len(cols)} Spalten
- VERFÜGBARE MERKMALE: {cols}{context}

### STATUS:
{warnung if warnung else "✅ Alle Kontext-Prüfungen bestanden."}

### AUFTRAG:
{veredelt}

### ANWEISUNG:
Nutze die oben genannten ram_facts für deine Antwort.
Ersetze alle Platzhalter wie '[df_name]' zwingend durch '{df_n}'.
Arbeite präzise.
"""
    return dossier

# soll text mit ist verbesserter text vergleichen um qualityt zu sichern
def frage_vergleich(text_user, text_veredelt, df_facts):
    """
    DYNAMISCHER RICHTER (Epic Reasoning - V12).
    Baut Logik-Pfade individuell nach Benutzeranfrage und ram_facts.
    Prüft Struktur, Kausalität und Willens-Sättigung.
    """
    import networkx as nx
    from pydantic import BaseModel, Field
    from typing import List, Optional

    # 1. WISSENS-ZUGRIFF (Tools & Phasen)
    tools_info = check_available_tools()
    vorgeschlagene_tools = [t for t in tools_info if t.lower() in text_veredelt.lower()]

    if not vorgeschlagene_tools:
        return False, "⚠️ NON-SENSE: Kein gültiges Tool in der Antwort erkannt."

    # 2. DYNAMISCHE LOGIK-GRAPH (Reasoning Engine)
    G = nx.DiGraph()
    ist_dreckig = df_facts.get('nan_count', 0) > 0
    ist_unverarbeitet = df_facts.get('categorical_count', 0) > 0
    for tool in vorgeschlagene_tools:
        phase = tools_info[tool]['phase']
        if phase == 'ML' and ist_dreckig:
            G.add_edge("CLEANING_REQUIRED", tool)
        elif phase == 'ML' and ist_unverarbeitet:
            G.add_edge("PREP_REQUIRED", tool)

        else:
            G.add_edge("START", tool)
    for tool in vorgeschlagene_tools:
        dependencies = list(G.predecessors(tool))

        if "CLEANING_REQUIRED" in dependencies:
            if not any(word in text_veredelt.lower() for word in ['clean', 'prep', 'reinigung', 'klib']):
                return False, f"⚠️ LOGIK-VETO: {tool} (ML) blockiert. Grund: Daten haben NaNs, aber kein Cleaning-Schritt im Text."

        if "PREP_REQUIRED" in dependencies:
            if not any(word in text_veredelt.lower() for word in ['encode', 'dummy', 'vector', 'prep']):
                return False, f"⚠️ LOGIK-VETO: {tool} (ML) blockiert. Grund: Nicht-numerische Daten ohne Vorverarbeitung."
    user_wille_eda = any(word in text_user.lower() for word in ['zeig', 'plot', 'visual', 'grafik', 'sehen'])
    hat_eda_tool = any(tools_info[t]['phase'] == 'EDA' for t in vorgeschlagene_tools)
    if user_wille_eda and not hat_eda_tool:
        return False, "⚠️ WILLENS-FEHLER: Du wolltest eine Visualisierung, aber die KI hat nur Text/Code ohne Grafik-Tool vorgeschlagen."

    # 5. ZERTIFIZIERUNG & RETURN
    #print(f"✅ Logik zertifiziert: {len(vorgeschlagene_tools)} Tool(s) geprüft und für logisch befunden.")
    return True, text_veredelt

# Verbindung der def funktionen
def frage(text_user, Language="DE"):
    """
    UNIVERSALER STRATEGISCHER VEREDLER (Orchestrator)
    LOGIK-KETTE:
    1. Sense ram_facts
    2. Pre-Verify (Logik-Check vorab -> Sättigung des Plans)
    3. Explain (Ollama schreibt mit Richter-Vorgabe)
    4. Post-Verify (Finale Zertifizierung)
    Logik-Priorität: Erst Logik-Graph, dann Generierung.
    """
    global llm
    if not pruefe_und_starte_ollama():
        return None
    start_time = time.time()
    stop_event = threading.Event()
    t = threading.Thread(target=start_spinner, args=(stop_event, Language=="DE"))
    t.start()
    try:
        name = hole_aktiven_dataframe_namen(suchbegriff=text_user)
        df_facts = extrahiere_ram_fakten(name) if name else {}
        logik_ok_vorab, hinweis_vorab = frage_vergleich(text_user, "PRE_CHECK_PLANUNG", df_facts)
        query = erstelle_optimierte_anfrage(text_user, df_facts)
        if not logik_ok_vorab:
            query += f"⚠️ LOGIK-VORGABE (RICHTER-VETO): {hinweis_vorab}"
            query += "INFO: Halte dich strikt an diese Phasen-Reihenfolge (Warenordnung)!"
        #print(query)
        antwort = llm.call(query, ram_facts=df_facts, Language=Language)
        logik_check_final, hinweis_final = frage_vergleich(text_user, antwort, df_facts)
        if not logik_check_final:
            korrektur_query = f"{query}❌ LOGIK-FEHLER IM ERSTEN VERSUCH: {hinweis_final}Bitte korrigiere die fachliche Kausalität!"
            antwort = llm.call(korrektur_query, ram_facts=df_facts, Language=Language)
        stop_event.set()
        t.join()
        dur = round(time.time() - start_time, 1)
        print(antwort)
        if 'TEMP_Clear_ML' in get_ipython().user_ns:
            get_ipython().user_ns['TEMP_Clear_ML'].append({
                "q": text_user,
                "a": antwort,
                "df": name,
                "logik": "Zertifiziert",
                "duration": dur
            })
        return

    except Exception as e:
        if 'stop_event' in locals(): stop_event.set()
        if 't' in locals() and t.is_alive(): t.join()
        print(f"❌ Fehler in der Logik-Kette (Klemmblock): {e}")
        return None


In [33]:
# Orchestrator
# standard einstellungen
def logic_setup(self):
    """
    Zuständigkeit: Zentrale Basis-Einstellungen (Setup).
    - Identität: Fixierung des Modells (llama3).
    - version forgabe welches modell das orchestrator hatt von LLM
    - Live-Logik: Keine RAM-Vorbelegung (Arbeit erfolgt via LivePacketLoader).
    - Hardware: Feste Ressourcen-Zuweisung für stabilen System-Call.
    """
    self.version = "v6"
    self.behavior_dir = "registry/agents"
    self.registry_path = "tool_logic_registry.json"
    self.config = {
        "temperature": 0.0,       # Absolute Fakten-Treue
        "num_thread": 4,          # Core-Reserve Standard (1 Core frei)
        "timeout": 1200,          # Ausfallsicherheit für große Pakete
        "num_ctx": 8192,          # Erweitertes Gedächtnis für Live-Daten/Tabellen
        "repeat_penalty": 1.1,    # Verhindert Logik-Schleifen/Wiederholungen
        "seed": 42,               # 100% identische Ergebnisse bei gleichem Input
        "top_p": 0.9              # Logische Konsistenz-Absicherung
    }
    if not os.path.exists(self.behavior_dir):
        os.makedirs(self.behavior_dir)

# Versions ermitlung
def type(self) -> str:
    """
    Zuständigkeit: Automatisches Auslesen der Setup-Identität.
    - Liest Modell (llama3) & Version (v6) direkt aus dem init.
    - Markiert den Live-Modus (L) für die df_ML Datenbank.
    - Erkennt Hardware-Drosselung (Threads) automatisch.
    """
    model = self.model_name_
    ver = self.version
    threads = self.config.get("num_thread", "X")
    return f"{model}_{ver}_L_T{threads}"

# BASIS: STABILER SYSTEM-CALL
def call_ollama_safe(self, payload):
    """
    Zuständigkeit: Dynamischer API-Aufruf mit automatischer Konfigurations-Übernahme.
    - Effizienz: Nutzt **self.config (Unpacking), um alle Setup-Werte blind zu übernehmen.
    - Flexibilität: Jede Änderung im init wird ohne Code-Anpassung sofort aktiv.
    - Sicherheit: Getrenntes Handling für Timeout und Stream-Stop.
    """
    payload["model"] = self.model_name_
    payload["stream"] = False
    payload["options"] = {**self.config}
    current_timeout = self.config.get("timeout", 1200)

    try:
        res = requests.post("http://localhost:11434/api/generate", json=payload, timeout=current_timeout)
        res.raise_for_status()
        return res.json().get("response", "").strip()

    except requests.exceptions.Timeout:
        return f"SYSTEM-STOP: Zeitüberschreitung nach {current_timeout}s (Live-Daten-Limit)."
    except Exception as e:
        return f"FEHLER im System-Call: {str(e)}"

# AGENTEN-AUDITION & EVOLUTION
def find_or_create_agent(self, instruction: str, detected_tools: list) -> dict:
    """
    Sucht fähige Agenten oder Ketten.
    Fokus: Strategische Planung ohne feste Namensbindung (Agnostisch).
    """
    p = BEHAVIOR_DIR
    if not os.path.exists(p): os.makedirs(p)
    all_files = [f for f in os.listdir(p) if f.endswith(('.yaml', '.yml'))]
    check_prompt = f"""
    AUFGABE: {instruction}
    BESTAND: {all_files}
    TOOLS: {detected_tools}

    ENTSCHEIDE:
    1. Passt ein Agent perfekt zur Logik der Frage? (Name nennen)
    2. Erfüllt eine KETTE ausreichend die Anfrage?
    3. Falls keine Deckung: 'EVOLUTION'.

    Antworte nur: 'MODE: [NAME oder KETTE oder EVOLUTION]'
    """
    decision = self.call_ollama_safe({"prompt": f"TASK: {instruction}\nFILES: {all_files}"})

    if "EVOLUTION" not in decision.upper():
        found_agents = [f for f in all_files if f in decision.replace('[','').replace(']','')]
        if found_agents:
            with open(os.path.join(p, found_agents[0]), 'r', encoding='utf-8') as f:
                return yaml.safe_load(f)
    gen_prompt = f"""
    ERSTELLE QUALITÄTS-AGENTEN-DNA (YAML).
    ZIEL: {instruction}
    WERKZEUGE: {detected_tools}

    ANFORDERUNG:
    1. STRATEGIE: Erstelle einen Master-Plan für die Tools {detected_tools}.
    2. LIVE-DATEN: Nutze KEINE festen Namen (kein df_Cleaning, kein df_sample).
       Der Plan muss auf das DataFrame wirken, das in der Query identifiziert wurde.
    3. REINHEIT: Keine Befehle zum Laden oder Speichern von Dateien, außer explizit gefordert.
    4. IMPORTS: Verlasse dich darauf, dass die Tool-Umgebung bereits geladen ist.

    FORMAT: instructions: | [Dein Master-Plan]
    """
    raw_dna = self.call_ollama_safe({"model": self.model_name_, "prompt": gen_prompt})
    try:
        clean_dna = raw_dna.replace("```yaml", "").replace("```", "").strip()
        new_dna = yaml.safe_load(clean_dna)
        if not isinstance(new_dna, dict) or 'instructions' not in new_dna:
            raise ValueError
    except:
        new_dna = {"instructions": f"Strategischer Plan für {instruction}. Nutze die verfügbaren Tools im RAM-Modus."}
    new_name = f"expert_v{len(all_files)+1}.yaml"
    with open(os.path.join(p, new_name), 'w', encoding='utf-8') as f:
        yaml.dump(new_dna, f)
    return new_dna

# imports zu LLM
def import_to_llm_bridge(self, sorted_tools, tools_status, instruction):
    """
    REINER BRÜCKENBAUER: Übergibt dem LLM das technische Wissen zur Tool-Einbindung.
    Keine Geister-Dateien, kein RAM-Ballast. Fokus auf korrekter Verbindung.
    """
    logic_blocks = []
    for tool in sorted_tools:
        info = tools_status.get(tool, {})
        if info.get('installed'):
            usage = info.get('usage_pattern', f"Importiere {tool} und nutze es für die Aufgabe.")
            logic_blocks.append(f"TECHNISCHE_VERBINDUNG ({tool}): {usage}")
    return "\n".join(logic_blocks)

def call(self, instruction: str, **kwargs) -> str:
    """
    DER AGNOSTISCHE ORCHESTRATOR:
    Verbindet Strategie (DNA) und Technik (Bridge) unter Einhaltung der Sprachvorgabe.
    """
    target_lang = kwargs.get('Language', 'DE').upper()
    lang_rule = "ANTWORTE STRENG AUF DEUTSCH." if target_lang == "DE" else "ANSWER STRICTLY IN ENGLISH."
    ram_facts = kwargs.get('ram_facts', "Kontext: Aktives DataFrame im RAM.")
    logic_veto = ""
    if "⚠️ LOGIK-VORGABE" in instruction or "❌ LOGIK-FEHLER" in instruction:
        logic_veto = "\n[KRITISCHE KORREKTUR-ANWEISUNG]: " + instruction.split("INFO:")[0] if "INFO:" in instruction else instruction
    detected_tools = [t for t in self.tools_status.keys() if t.upper() in instruction.upper()]
    sorted_tools = sorted(detected_tools, key=lambda x: self.tools_status[x].get('priority', 5))
    syntax_bridge = self.import_to_llm_bridge(sorted_tools)
    agent_dna = self.find_or_create_agent(instruction, sorted_tools)
    prompt = f"""
    [HAUPT-VORGABE: {lang_rule}]
    [STRATEGIE-DNA: {agent_dna.get('instructions')}]
    [IMPORT-BRÜCKE: {syntax_bridge}]
    [DATEN-QUELLE: {ram_facts}]
    AUFTRAG: {instruction}

    STRIKTE ARBEITS-REGELN:
    1. SPRACHE: {lang_rule} (Kein Englisch!)
    2. DATEN: Nutze NUR die Quelle aus 'DATEN-QUELLE'. Kein df_sample, kein df_Cleaning!
    3. TOOLS: Nutze die 'IMPORT-BRÜCKE' für die Einbindung der Werkzeuge.
    4. MODUS: Arbeite rein im RAM. Erzeuge keine neuen CSV- oder Excel-Dateien.
    5. KONSISTENZ: Prüfe, ob die Tool-Ergebnisse zur Logik der Frage passen.
    """
    output = self.call_ollama_safe({"model": self.model_name_, "prompt": prompt})
    gc.collect()
    return output

# verbinde alle def unter classe
class OllamaOrchestrator:
    def __init__(self, model_name="llama3"):
        # Das Modell wird GENAU HIER EINMAL gesetzt
        self.model_name_ = model_name
        self.tools_status = {}
        logic_setup(self)

    def type(self) -> str:
        t = self.config.get("num_thread", "X")
        return f"{self.model_name_}_{self.version}_L_T{t}"

    def call_ollama_safe(self, payload):
        return call_ollama_safe(self, payload)

    def find_or_create_agent(self, instruction, detected_tools):
        return find_or_create_agent(self, instruction, detected_tools)

    def import_to_llm_bridge(self, sorted_tools):
        return import_to_llm_bridge(self, sorted_tools, self.tools_status)

    def call(self, instruction, **kwargs):
        return call(self, instruction, **kwargs)

llm = OllamaOrchestrator(model_name="llama3")

1. Nutzen Sie LLM für erste Erkundungen
Laden den Datensatz, nutzen den integrierten KI-Assistenten LLM, um Ihr Projekt zu starten. Bitten Sie LLM, den Datensatz zu beschreiben und wichtige Merkmale wie Spaltennamen, Datentypen und fehlende Werte zusammenzufassen.
Diese automatisierte Analyse hilft Ihnen, die Struktur des Datensatzes schnell zu erfassen und zu entscheiden, welche Spalten für Ihre Ziele am relevantesten sind.

In [ ]:
frage("mach mit cleaning_pre, Datensatz beschreiben in dem die wichtige Merkmale wie Spaltennamen, Datentypen und fehlende Werte zusammenzufassen")

2. Visualisieren Sie die Daten mit AutoViz.
Als Nächstes auf das visuelle Verständnis der Daten. Mit AutoViz können Diagramme und Grafiken erstellen, die wichtige Erkenntnisse über die Daten liefern. Vergleichen die Ergebnisse von AutoViz mit den vorherigen Vorschlägen von LLM.

In [ ]:
frage('mit pandasai Als Nächstes auf das visuelle Verständnis der Daten. Mit AutoViz können Diagramme und Grafiken erstellen, die wichtige Erkenntnisse über die Daten liefern. Vergleichen die Ergebnisse von AutoViz mit den vorherigen Vorschlägen von LLM.')

In [34]:
frage('wie fiel mb hatt der df_sample.csv')

❌ Fehler in der Logik-Kette (Klemmblock): 'list' object has no attribute 'model_name_'              


3. Stimmungsanalyse mithilfe von Hugging Face LLMs
Wenden abschließend ein Large Language Model (LLM) von Hugging Face an, um die textSpalte des Datensatzes zu analysieren. Diese Spalte enthält den vollständigen Tweet des Kunden. Mithilfe des Modells bewerten Sie die Stimmung der Nachrichten.
Sie können entweder Folgendes verwenden:
- Die Hugging Face Transformers-Bibliothek oder
- die Inference API ermöglicht den Zugriff, ohne Modelle herunterladen zu müssen.

# Präsentationskontext
Dieses Projekt wird im Rahmen einer Live-Präsentation vorgestellt, nachdem alle drei Projekte zur Steigerung der KI-gestützten Produktivität abgeschlossen sind.
Im Rahmen der Präsentation werden Sie Folgendes erläutern:
- Das analytische Ziel dieses Projekts
- Wie LLM, AutoViz und Sprachmodelle verwendet wurden
- Wichtige Erkenntnisse zur Stimmungslage aufgedeckt
- Was Sie über die Verwendung von KI für textbasierte Analysen gelernt haben

Der Schwerpunkt der Präsentation liegt auf der Interpretation und der Steigerung der Produktivität, nicht auf technischen Details der Umsetzung.

# Qualitätserwartungen
- KI-Werkzeuge sollten gezielt und nicht oberflächlich eingesetzt werden.
- Die Ergebnisse müssen klar interpretiert und in den Kontext gesetzt werden.
- Die Reflexionen sollten ein Verständnis sowohl der Daten als auch der Werkzeuge erkennen lassen.
- Die gewonnenen Erkenntnisse sollten auf den im Rahmen der Analyse gewonnenen Erkenntnissen beruhen.